In [47]:
# Preparing pathing
from titanic_ml import paths
import matplotlib.pyplot as plt
from titanic_ml.common.data.eda import quick_description, summarize_categorical_column, summarize_numerical_column, deep_describe


ImportError: cannot import name 'summarize_categorical_column' from 'titanic_ml.common.data.eda' (C:\Users\jonma\Documents\GitHub\Machine_learning\Titanic\titanic_ml\common\data\eda.py)

In [ ]:
from titanic_ml.data import load_train_data

df = load_train_data()
df.head()

In [ ]:
from titanic_ml.common.data.eda import quick_description
TARGET = "Survived"

# EDA: Exploratory Data Analysis
quick_description(df)

In [ ]:
# Length of string columns
length = df['Embarked'].str.len().describe()
print(f"Length of 'Embarked':\n{length}")

'''
This reveals:

free text
codes
IDs
malformed entries

Example:

all length 2 → state codes
all length 36 → UUIDs
huge variance → real text

Very powerful for schema discovery.
'''


# Cardinality: number of unique values / total rows
cardinality = df['Embarked'].nunique(dropna=True) / len(df['Embarked'])
print(f"Cardinality of 'Embarked': {cardinality:.2%}")

# Ratio	Meaning
'''
Very low = categorical
Medium = mixed
Very high = ID-like/free text

Example:

gender → 0.00002
country → 0.01
customer_id → 1.0

This immediately reveals:

IDs
UUIDs
text fields
columns unsuitable for one-hot encoding
'''

# Dominance Ratio
top_freq = df['Embarked'].value_counts(dropna=False, normalize=True).iloc[0]
print(f"Dominance Ratio of 'Embarked': {top_freq:.2%}")

'''
Dominance Ratio	Meaning
Very low = no dominant category
Medium = some dominance
Very high = one category dominates

useful for detecting useless columns.

If:

one value = 99%
column may be useless

Example:
"ACTIVE" repeated everywhere
'''

print()
print()
# Memory usage
df.memory_usage(deep=True)
print(f"Total memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

# Column health summary
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'missing_%': df.isna().mean() * 100,
    'unique': df.nunique(),
    'cardinality_%': df.nunique() / len(df) * 100,
    'top_value': df.apply(lambda s: s.value_counts(dropna=False).idxmax()),
    'dominance_%': df.apply(lambda s: s.value_counts(dropna=False, normalize=True).iloc[0] * 100),
    'bottom_value': df.apply(lambda s: s.value_counts(dropna=False).idxmin()),
})

summary = summary.sort_values(
    by='missing_%',
    ascending=False
)

print(summary)


Length of 'Embarked':
count    889.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: Embarked, dtype: float64
Cardinality of 'Embarked': 0.34%
Dominance Ratio of 'Embarked': 72.28%


Total memory usage: 315.03 KB
               dtype  missing_%  unique  cardinality_%  \
Cabin         object  77.104377     147      16.498316   
Age          float64  19.865320      88       9.876543   
Embarked      object   0.224467       3       0.336700   
PassengerId    int64   0.000000     891     100.000000   
Survived       int64   0.000000       2       0.224467   
Pclass         int64   0.000000       3       0.336700   
Name          object   0.000000     891     100.000000   
Sex           object   0.000000       2       0.224467   
SibSp          int64   0.000000       7       0.785634   
Parch          int64   0.000000       7       0.785634   
Ticket        object   0.000000     681      76.430976   
Fare         float64   0.000

In [ ]:
import seaborn as sns
# Missing Values Heatmap
sns.heatmap(df.isna(), cbar=False)

'''
This immediately shows:
missingness structure
grouped missing values
data pipeline problems
'''
plt.show()

In [ ]:
# Missing Percentage Barplot
missing = df.isna().mean().sort_values(ascending=False)

missing[missing > 0].plot.bar()

In [ ]:
# cardinality barplot
df.nunique().sort_values().plot.barh()

In [54]:
def summarize_categorical_column(series, max_display=20):
    # Count frequencies including NaN
    counts = series.value_counts(dropna=False)

    # Percentages
    percents = (
        series.value_counts(dropna=False, normalize=True) * 100
    ).round(2)

    # Build summary table
    summary = pd.DataFrame({
        'count': counts,
        'percent': percents
    })

    # Renaming NA/None to 'MISSING' for better readability
    summary = summary.rename(index={pd.NA: 'MISSING', None: 'MISSING'})

    # If small enough, show everything
    if len(summary) <= max_display:
        return summary

    # Otherwise show top/bottom
    top_n = max_display // 2

    truncated = pd.concat([
        summary.head(top_n),
        pd.DataFrame(
            {'count': ['...'], 'percent': ['...']},
            index=['...']
        ),
        summary.tail(top_n)
    ])

    return truncated

def summarize_numerical_column(df):
    return df.describe(include=['number'])

def deep_describe(df):

    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    numerical_cols = df.select_dtypes(include=['number']).columns

    print('Categorical columns summary:')
    for col in categorical_cols:
        print(f'\n=== {col} ===')
        print(summarize_categorical_column(df[col]))
    
    print('Numerical columns summary:')
    for col in numerical_cols:
        print(f'\n=== {col} ===')
        print(summarize_numerical_column(df[col]))

def summarize_dataframe(df):
    summary = pd.DataFrame({
    'dtype': df.dtypes,
    'missing_%': df.isna().mean() * 100,
    'unique': df.nunique(),
    'cardinality_%': df.nunique() / len(df) * 100,
    'top_value': df.apply(lambda s: s.value_counts(dropna=False).idxmax()),
    'dominance_%': df.apply(lambda s: s.value_counts(dropna=False, normalize=True).iloc[0] * 100),
    'bottom_value': df.apply(lambda s: s.value_counts(dropna=False).idxmin()),
    })

    summary = summary.sort_values(
        by='missing_%',
        ascending=False
    )

    print(summary)

summarize_dataframe(df)
deep_describe(df)

               dtype  missing_%  unique  cardinality_%  \
Cabin         object  77.104377     147      16.498316   
Age          float64  19.865320      88       9.876543   
Embarked      object   0.224467       3       0.336700   
PassengerId    int64   0.000000     891     100.000000   
Survived       int64   0.000000       2       0.224467   
Pclass         int64   0.000000       3       0.336700   
Name          object   0.000000     891     100.000000   
Sex           object   0.000000       2       0.224467   
SibSp          int64   0.000000       7       0.785634   
Parch          int64   0.000000       7       0.785634   
Ticket        object   0.000000     681      76.430976   
Fare         float64   0.000000     248      27.833895   

                           top_value  dominance_%             bottom_value  
Cabin                            NaN    77.104377                      A23  
Age                              NaN    19.865320                     34.5  
Embarked      

In [55]:
import pandas as pd


def format_missing_index(index):
    return ["MISSING" if pd.isna(idx) else idx for idx in index]


def summarize_categorical_column(series, max_display=20):
    counts = series.value_counts(dropna=False)
    percents = (series.value_counts(dropna=False, normalize=True) * 100).round(2)

    summary = pd.DataFrame({
        "count": counts,
        "percent": percents,
    })

    summary.index = format_missing_index(summary.index)

    if len(summary) <= max_display:
        return summary

    top_n = max_display // 2

    truncated = pd.concat([
        summary.head(top_n),
        pd.DataFrame(
            {"count": ["..."], "percent": ["..."]},
            index=["..."],
        ),
        summary.tail(top_n),
    ])

    return truncated


def summarize_numerical_column(series):
    summary = series.describe()

    extra = pd.Series({
        "missing_count": series.isna().sum(),
        "missing_%": round(series.isna().mean() * 100, 2),
        "skew": series.skew(),
        "kurtosis": series.kurtosis(),
    })

    return pd.concat([summary, extra])


def safe_top_value(series):
    counts = series.value_counts(dropna=False)
    if counts.empty:
        return None
    value = counts.index[0]
    return "MISSING" if pd.isna(value) else value


def safe_bottom_value(series):
    counts = series.value_counts(dropna=False)
    if counts.empty:
        return None
    value = counts.index[-1]
    return "MISSING" if pd.isna(value) else value


def summarize_dataframe(df):
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null_count": df.notna().sum(),
        "missing_count": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=False),
        "cardinality_%": (df.nunique(dropna=False) / len(df) * 100).round(2),
        "top_value": df.apply(safe_top_value),
        "dominance_%": df.apply(
            lambda s: round(s.value_counts(dropna=False, normalize=True).iloc[0] * 100, 2)
            if not s.empty else None
        ),
        "bottom_value": df.apply(safe_bottom_value),
    })

    return summary.sort_values(by="missing_%", ascending=False)


def deep_describe(df, max_display=20):
    categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns
    numerical_cols = df.select_dtypes(include=["number"]).columns

    print("Categorical columns summary:")
    for col in categorical_cols:
        print(f"\n=== {col} ===")
        print(summarize_categorical_column(df[col], max_display=max_display))

    print("\nNumerical columns summary:")
    for col in numerical_cols:
        print(f"\n=== {col} ===")
        print(summarize_numerical_column(df[col]))

In [57]:
print(summarize_dataframe(df))
deep_describe(df)

               dtype  non_null_count  missing_count  missing_%  unique  \
Cabin         object             204            687      77.10     148   
Age          float64             714            177      19.87      89   
Embarked      object             889              2       0.22       4   
PassengerId    int64             891              0       0.00     891   
Survived       int64             891              0       0.00       2   
Pclass         int64             891              0       0.00       3   
Name          object             891              0       0.00     891   
Sex           object             891              0       0.00       2   
SibSp          int64             891              0       0.00       7   
Parch          int64             891              0       0.00       7   
Ticket        object             891              0       0.00     681   
Fare         float64             891              0       0.00     248   

             cardinality_%           